# project_22_conformational_switch — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — multi-state theory + the two-state hello-world

**Standard slot:** *define & explore.* **For Project 22 this means:** understand what a
conformational switch *is* (ONE sequence compatible with TWO backbones, toggled by a trigger), pin
down the metrics, define your two states + trigger, and run the **mock hello-world** end-to-end:
one shared sequence → predict **both** states → compute the energy gap (D0).

Run `00_setup.ipynb` first in this session. The mock backend runs anywhere (no GPU); switch to the
real RFdiffusion/MPNN/AF2 backends on Colab/A100.

## What a conformational switch is (and why it's hard)

A switch is **one amino-acid sequence that is compatible with two distinct backbones** — state A and
state B — and interconverts between them when a **trigger** fires (pH / ligand / light / temperature).
The Baker lab's **LOCKR** showed de novo switches are possible, but they are rare:

- a sequence that fits state A well usually fits state B **badly**, and
- our structure predictors (AF2) typically return a **single** dominant state, so even *confirming*
  the switch in silico is genuinely hard.

So the whole project is a search for the unusual sequence that satisfies **both** states, plus an
honest account of how often *anything* does.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| per-state scRMSD | Å | designed-vs-predicted Cα-RMSD for ONE state (< 2 Å = foldable to that shape) | the protein toggles / state is populated |
| per-state pLDDT | 0–100 | local confidence of that state's refold | thermostability / ΔG / "this state exists" |
| state energy gap | relative units (**NOT kcal/mol**) | how far apart the two states sit (proxy from per-state fit) | a real ΔΔG |
| per-state MPNN score | — | multi-state MPNN's fit to each backbone (lower = better) | binding/function |

A switch must satisfy **per-state scRMSD < 2 Å for A AND B** *and* land **inside** the energy-gap
band `[SWITCH_GAP_MIN, SWITCH_GAP_MAX]`: too large a gap ⇒ the high-energy state is never populated
(no switch); too small ⇒ the states are indistinct (no defined OFF/ON). Write your own one-paragraph
definitions in `D0`, including the "does not mean" column — that is where the published mistakes live.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Define your two states + trigger

Edit `data/inputs/two_state_def.txt` (the Phase-0 template) with your **two states + trigger +
measurable success criteria + controls**. The notebook reads the **topology** and **trigger** from
there; the two backbones themselves are *generated* in notebook 02. Below we just transcribe the
two knobs the campaign needs (keep them in sync with the file).

In [ ]:
import multistate_tools as ms

# Transcribe these from data/inputs/two_state_def.txt (your D0 problem statement).
TOPOLOGY = "hinge"     # one of ms.TOPOLOGIES
TRIGGER  = "pH"        # one of ms.TRIGGERS
LENGTH   = 100         # same length in both states (tied multi-state design)

assert TOPOLOGY in ms.TOPOLOGIES, ms.TOPOLOGIES
assert TRIGGER in ms.TRIGGERS, ms.TRIGGERS
print("topologies:", ms.TOPOLOGIES)
print("triggers   :", ms.TRIGGERS)
print(f"this design: {TRIGGER}-triggered toggle between two {TOPOLOGY} states, L={LENGTH}")
print("per-state bars: scRMSD <", ms.SELF_CONSISTENT_SCRMSD, "A, pLDDT >=", ms.SWITCH_PLDDT)
print("switchable gap band:", (ms.SWITCH_GAP_MIN, ms.SWITCH_GAP_MAX), "(relative units, NOT kcal/mol)")

## Mock hello-world: one sequence → two states → energy gap

Using the **deterministic mock backend** (synthetic, labelled `EXAMPLE_DATA` — **never** present as
real designs), we exercise the entire plumbing: generate the two state backbones, design ONE shared
sequence with multi-state MPNN, predict **both** states from that one sequence, and compute the
energy gap. On Colab/A100 you flip `tool="mock"` → the real backends.

In [ ]:
# 1) the two state backbones (mock: placeholders; on Colab -> RFdiffusion x2)
tsd = ms.generate_two_states(topology=TOPOLOGY, trigger=TRIGGER, tool="mock", length=LENGTH, seed=0)
print("state A:", tsd.state_a.state, tsd.state_a.topology, "L=", tsd.state_a.length, "| synthetic=", tsd.state_a.synthetic)
print("state B:", tsd.state_b.state, tsd.state_b.topology, "L=", tsd.state_b.length, "| synthetic=", tsd.state_b.synthetic)
print("trigger:", tsd.trigger, "-", tsd.trigger_detail)

# 2) ONE shared sequence compatible with BOTH backbones (mock multi-state MPNN)
seq = ms.multistate_mpnn(tsd.state_a, tsd.state_b, n=1, tool="mock", seed=0)[0]
print("\nshared sequence:", seq.design_id)
print("  per-state MPNN score  A:", seq.mpnn_score_a, " B:", seq.mpnn_score_b, "(lower = better fit)")

# 3) predict BOTH states from that ONE sequence
pa = ms.af2_predict_state(seq.sequence, tsd.state_a, tool="mock")
pb = ms.af2_predict_state(seq.sequence, tsd.state_b, tool="mock")
print("\nstate A  scRMSD:", pa.scrmsd_to_state, "A  pLDDT:", pa.plddt)
print("state B  scRMSD:", pb.scrmsd_to_state, "A  pLDDT:", pb.plddt)

# 4) the energy gap between the two predicted states
eg = ms.energy_gap(pa, pb)
print("\nenergy gap:", eg.gap, "| favored:", eg.favored_state, "| switchable:", eg.switchable)
print("note:", eg.note)
print("\n[mock = SYNTHETIC EXAMPLE_DATA — not a real switch]")

### Read the hello-world honestly

Notice (in the synthetic numbers) the pattern the real problem has: state A — the "designed-for"
state — tends to fit better than state B, and the gap is small and noisy. That is the multi-state
difficulty in miniature. A *real* candidate must pass **both** per-state bars **and** sit inside the
switchable band — and you must still worry that AF2 only ever modeled one of the states.

## Visualize a state (py3Dmol)
Once you have real predicted PDBs (notebook 04), eyeball state A and state B side by side.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=500, height=400)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after real predictions):
# show_pdb("results/two_state/state_A_pred.pdb")
# show_pdb("results/two_state/state_B_pred.pdb")
print("show_pdb(pdb_path) ready — compare state A vs state B once you have real models.")

## D0 checklist
- [ ] One-paragraph definition of **per-state scRMSD** and the **energy gap** — *with* what each does **not** mean (gap ≠ ΔΔG; low scRMSD ≠ the switch toggles).
- [ ] `data/inputs/two_state_def.txt` filled in: two states + trigger + **measurable** success criteria + the three controls.
- [ ] Reproduced hello-world: one shared sequence → two per-state predictions → energy gap (screenshot/printout).
- [ ] Every accession in `data/README.md` verified on RCSB.
- [ ] `LOG.md` entry: date, backend (mock here), seed.

**Next:** `02_generate.ipynb` — generate the two backbones + multi-state MPNN shared-sequence pool.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Generate — two state backbones + multi-state MPNN shared-sequence pool

**Standard slot:** *design campaign.* **For Project 22 this means:** generate the **two state
backbones** (RFdiffusion ×2), then run **multi-state ProteinMPNN** (residue identities *tied* across
both backbones) to search for sequences compatible with **both** states, predict both states, and
assemble `results/multistate_designs.csv` (D2).

**Diversity before filtering** — generate a real pool here; you filter in notebook 03. The mock
backend runs anywhere; switch to the real backends on Colab/A100.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change)

The real backends live in fast-moving repos. Before a real run, confirm the pinned upstreams still
exist (HTTP `HEAD`). **Pin the exact commit/tag** in `scripts/multistate_tools.py` and `LOG.md` — the
status check below confirms reachability, *not* that your pinned commit is unchanged.

In [ ]:
import requests

# Pinned upstreams for the multi-state campaign (replace the comment with the COMMIT/TAG you pin):
UPSTREAMS = {
    "RFdiffusion": "https://github.com/RosettaCommons/RFdiffusion",   # pin a commit/tag in multistate_tools.py
    "ColabDesign": "https://github.com/sokrypton/ColabDesign",        # RFdiffusion + tied-MPNN driver; pin a commit
    "ProteinMPNN": "https://github.com/dauparas/ProteinMPNN",         # run in tied/multi-state mode; pin a commit
    "OpenMM":      "https://github.com/openmm/openmm",                # (extension) transition MD; pin a release
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"{name:12s} {r.status_code}  {url}")
    except Exception as e:  # noqa: BLE001
        print(f"{name:12s} UNREACHABLE  {url}  ({e})")
print("\n(Pin the exact commit/tag you use; log it. Reachability != version unchanged.)")

## 1 · Generate the two state backbones (RFdiffusion ×2)

State A and state B are two related-but-distinct backbones the **same** sequence must adopt. Two
common routes (see `MANUAL.md §2`): (a) generate A, then B as a conformational variant (partial
diffusion / hinge re-fold of A), or (b) build de novo backbones matching a known two-state template
pair (LOCKR latch/cage; an open/closed hinge). **This is two RFdiffusion runs — A100/HPC for the
campaign; a T4 runs only a tiny fallback.**

In [ ]:
import multistate_tools as ms

TOPOLOGY = "hinge"     # keep in sync with data/inputs/two_state_def.txt
TRIGGER  = "pH"
LENGTH   = 100
SEED     = 0

# tool="mock" -> placeholders (no GPU); on Colab use tool="rfdiffusion" (two backbone runs).
tsd = ms.generate_two_states(topology=TOPOLOGY, trigger=TRIGGER, tool="mock",
                             length=LENGTH, seed=SEED, out_dir="results/two_state")
print("state A:", tsd.state_a.as_row())
print("state B:", tsd.state_b.as_row())
print("trigger:", tsd.trigger, "-", tsd.trigger_detail)

## 2 · Multi-state ProteinMPNN — one sequence for BOTH backbones

Tie residue identities across the two backbones and optimize one shared sequence under **both**
states. Record the **per-state MPNN score** for A and B *separately* — a good switch fits BOTH, not
just one. Sample a pool (`N_DESIGNS`) so there is diversity to filter later. A single sequence
satisfying two states well is **rare**: expect a low yield (see `MANUAL.md §1`).

In [ ]:
N_DESIGNS = 24     # campaign scale; on a T4 fallback keep this small

# tool="mock" -> deterministic synthetic pool; on Colab use tool="proteinmpnn" (tied mode).
shared = ms.multistate_mpnn(tsd.state_a, tsd.state_b, n=N_DESIGNS, tool="mock", seed=SEED)
print(len(shared), "shared sequences (multi-state MPNN)")
for s in shared[:3]:
    print(" ", s.design_id, "| A score", s.mpnn_score_a, "| B score", s.mpnn_score_b,
          "| len", len(s.sequence))
print("[mock = SYNTHETIC EXAMPLE_DATA — not real designs]")

## 3 · Predict BOTH states for every shared sequence → results CSV

For each shared sequence, predict it toward **state A** and toward **state B** (AF2 for trusted
picks, ESMFold for triage), compute the per-state scRMSD/pLDDT, and the energy gap. Assemble one row
per design into `results/multistate_designs.csv` — the D2 artifact that notebooks 03/04 consume.

In [ ]:
import pandas as pd

rows = []
for s in shared:
    if not s.sequence:
        continue
    pa = ms.af2_predict_state(s.sequence, tsd.state_a, tool="mock")   # -> esmfold/af2 on Colab
    pb = ms.af2_predict_state(s.sequence, tsd.state_b, tool="mock")
    eg = ms.energy_gap(pa, pb)
    rows.append(dict(
        design_id=s.design_id, sequence=s.sequence, topology=TOPOLOGY, trigger=TRIGGER, seed=SEED,
        mpnn_score_a=s.mpnn_score_a, mpnn_score_b=s.mpnn_score_b,
        scrmsd_a=pa.scrmsd_to_state, plddt_a=pa.plddt,
        scrmsd_b=pb.scrmsd_to_state, plddt_b=pb.plddt,
        energy_gap=eg.gap, favored_state=eg.favored_state, switchable=eg.switchable,
        synthetic=bool(s.synthetic), tool="mock"))

df = pd.DataFrame(rows)
df.to_csv("results/multistate_designs.csv", index=False)
print("wrote results/multistate_designs.csv", df.shape)
df.head()

### A first look at the A-vs-B trade-off
Even on synthetic data you should see the prior the real problem has: sequences that fit A well tend
to fit B worse, only a few balance both, and the gap is small/noisy. Quantify how often *anything*
satisfies both states — that fraction is the story of the interim report.

In [ ]:
import numpy as np
pass_a = (df["scrmsd_a"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_a"] >= ms.SWITCH_PLDDT)
pass_b = (df["scrmsd_b"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_b"] >= ms.SWITCH_PLDDT)
print(f"N total              : {len(df)}")
print(f"pass state A (mock)  : {int(pass_a.sum())}")
print(f"pass state B (mock)  : {int(pass_b.sum())}")
print(f"pass BOTH (mock)     : {int((pass_a & pass_b).sum())}")
print(f"switchable gap (mock): {int(df['switchable'].fillna(False).sum())}")
print("\n[SYNTHETIC EXAMPLE_DATA — illustrates the trade-off, not a real hit rate]")

## D2 checklist
- [ ] Two state backbones generated (RFdiffusion ×2) — note the GPU (A100/HPC for the real run).
- [ ] `results/multistate_designs.csv`: one row per design with per-state scrmsd/pLDDT, per-state MPNN scores, energy gap, seed.
- [ ] Per-state MPNN scores recorded for **A and B separately** (the trade-off).
- [ ] Version-verify cell run; pinned commits/tags + seeds in `LOG.md`.
- [ ] 3–4 page interim report on the early A-vs-B trade-off and how often anything satisfies both states.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on **both** states.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared multi-layer filter, on BOTH states

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 22** a switch must fold to **both** states, so you build **two**
`fp.Design` objects per design (one for state A, one for state B), each with
`design_type="monomer"`, and a design only counts as a switch candidate if it passes the monomer
foldability bar for **A AND B** (D3 part 1).

Run `00`–`02` first so `results/multistate_designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. We apply
its **monomer** cutoffs to each state independently.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nApplying design_type='monomer' to EACH of the two states.")

## Build `fp.Design` objects for BOTH states

For each row of `multistate_designs.csv` we create two `fp.Design` records — `..._A` and `..._B` —
populating each state's per-state `scrmsd` and `plddt`. We then run the pipeline **once per state**
and intersect: a switch candidate must pass **A AND B**.

In [ ]:
df = pd.read_csv("results/multistate_designs.csv")

designs_a, designs_b = [], []
for _, r in df.iterrows():
    designs_a.append(fp.Design(
        design_id=f"{r['design_id']}__A", sequence=str(r["sequence"]),
        design_type="monomer", scrmsd=r.get("scrmsd_a"), plddt=r.get("plddt_a"),
        extra={"state": "A", "energy_gap": r.get("energy_gap"), "switchable": r.get("switchable")}))
    designs_b.append(fp.Design(
        design_id=f"{r['design_id']}__B", sequence=str(r["sequence"]),
        design_type="monomer", scrmsd=r.get("scrmsd_b"), plddt=r.get("plddt_b"),
        extra={"state": "B", "energy_gap": r.get("energy_gap"), "switchable": r.get("switchable")}))
print(len(designs_a), "state-A Design objects |", len(designs_b), "state-B Design objects")

## Run the pipeline on each state

We use layer 1 (self-consistency: scRMSD + pLDDT) since the campaign CSV carries per-state scRMSD and
pLDDT (no orthogonal/physics columns yet — add those layers once you compute them). `fp.report`
prints the hit-rate accounting and the survival-at-each-layer figure for **each** state.

In [ ]:
ranked_a = fp.run_pipeline(designs_a, design_type="monomer", use_layers=(1,))
ranked_b = fp.run_pipeline(designs_b, design_type="monomer", use_layers=(1,))

print("===== STATE A =====")
top_a = fp.report(ranked_a, top_n=10, save_prefix="results/proj22_state_A")
print("\n===== STATE B =====")
top_b = fp.report(ranked_b, top_n=10, save_prefix="results/proj22_state_B")
ranked_a.to_csv("results/ranked_state_A.csv", index=False)
ranked_b.to_csv("results/ranked_state_B.csv", index=False)
print("\nwrote results/ranked_state_A.csv and results/ranked_state_B.csv")

## Intersect: a switch candidate passes A **AND** B

This is the move that makes it a *switch* filter rather than two monomer filters. Pass requires
`layers_passed >= 1` for **both** states (and, for a real switch, the energy gap inside the band).

In [ ]:
pa = ranked_a.set_index("design_id")["layers_passed"]
pb = ranked_b.set_index("design_id")["layers_passed"]

both = []
for _, r in df.iterrows():
    ok_a = int(pa.get(f"{r['design_id']}__A", 0)) >= 1
    ok_b = int(pb.get(f"{r['design_id']}__B", 0)) >= 1
    both.append(dict(design_id=r["design_id"], pass_a=ok_a, pass_b=ok_b,
                     pass_both=ok_a and ok_b, energy_gap=r.get("energy_gap"),
                     switchable=bool(r.get("switchable")) if pd.notna(r.get("switchable")) else False))
switch_df = pd.DataFrame(both)
switch_df.to_csv("results/switch_candidates.csv", index=False)

n = len(switch_df)
print(f"N total            : {n}")
print(f"pass state A       : {int(switch_df['pass_a'].sum())}")
print(f"pass state B       : {int(switch_df['pass_b'].sum())}")
print(f"pass BOTH (switch) : {int(switch_df['pass_both'].sum())}")
print(f"  ...and switchable: {int((switch_df['pass_both'] & switch_df['switchable']).sum())}")
print("\nThe honest hit-rate ladder: N(A) / N(B) / N(BOTH) / N(switchable).")
print("[mock numbers are SYNTHETIC EXAMPLE_DATA]")
switch_df.head(10)

## D3 (part 1) checklist
- [ ] `fp.Design` built for **both** states; `design_type="monomer"` for each.
- [ ] `fp.run_pipeline(..., design_type="monomer")` + `fp.report(...)` run **per state** (survival-at-each-layer for A and B).
- [ ] Switch candidates = pass A **AND** B, saved to `results/switch_candidates.csv`.
- [ ] Honest hit-rate ladder reported: N(A) / N(B) / N(BOTH) / N(switchable).

**Next:** `04_validate.ipynb` — AF2 predicts both states from one sequence + the energy-gap study + single- vs multi-state benchmark.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — both states from one sequence + energy gap + the benchmark

**Standard slot:** *validate (in silico).* **For Project 22 this is the core science:** confirm one
sequence can fold to **both** states (and worry that AF2 may only show one), characterize the
**energy-gap distribution**, and run the **single- vs multi-state benchmark** — the contrast that
turns this from a demo into a study (D3 part 2).

Needs `results/multistate_designs.csv` (from notebook 02). Mock backend runs anywhere.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · AF2 predicts BOTH states from one sequence

For trusted picks, predict the shared sequence toward **each** state and compare per-state scRMSD.
**The central caveat:** AF2 returns a single dominant state and may not capture both — so where
possible **bias prediction toward each state** (templates / initial guess) and report the unbiased
*and* state-biased results. A "switch" claimed from one unbiased model is not a switch.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import multistate_tools as ms

df = pd.read_csv("results/multistate_designs.csv")

# per-state foldability (uses the project's bars)
df["pass_a"] = (df["scrmsd_a"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_a"] >= ms.SWITCH_PLDDT)
df["pass_b"] = (df["scrmsd_b"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_b"] >= ms.SWITCH_PLDDT)
df["pass_both"] = df["pass_a"] & df["pass_b"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df["scrmsd_a"], df["scrmsd_b"], c=df["pass_both"].map({True: "tab:green", False: "tab:gray"}))
ax.axvline(ms.SELF_CONSISTENT_SCRMSD, ls="--", c="k", lw=0.8)
ax.axhline(ms.SELF_CONSISTENT_SCRMSD, ls="--", c="k", lw=0.8)
ax.set_xlabel("state A scRMSD (A)"); ax.set_ylabel("state B scRMSD (A)")
ax.set_title("Per-state self-consistency (green = passes BOTH)")
plt.tight_layout(); plt.savefig("results/per_state_scrmsd.png", dpi=150); plt.show()
print("green points pass BOTH states; lower-left quadrant = the switch candidates.")
print("[SYNTHETIC EXAMPLE_DATA — AF2 may not actually produce both states; see the caveat]")

## 2 · The state energy-gap distribution

A switch needs the gap **inside the band**: close enough to interconvert, distinct enough for a
defined OFF/ON. Plot the distribution and shade the switchable band. **The gap is a teaching-grade
proxy in relative units — NOT a ΔΔG**; a "switchable" flag is a hypothesis, not a measurement.

In [ ]:
gaps = df["energy_gap"].dropna().values
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(gaps, bins=15, color="tab:blue", alpha=0.8)
ax.axvspan(ms.SWITCH_GAP_MIN, ms.SWITCH_GAP_MAX, color="tab:green", alpha=0.2,
           label=f"switchable band [{ms.SWITCH_GAP_MIN}, {ms.SWITCH_GAP_MAX}]")
ax.set_xlabel("state energy gap (relative units, NOT kcal/mol)")
ax.set_ylabel("designs"); ax.set_title("State energy-gap distribution")
ax.legend(); plt.tight_layout(); plt.savefig("results/energy_gap_dist.png", dpi=150); plt.show()
n_switch = int(df["switchable"].fillna(False).sum())
print(f"{n_switch} / {len(df)} designs fall inside the switchable band (mock).")
print("CAVEAT: relative-units proxy from prediction fit, not a free energy. Confirm with physics/MD.")

## 3 · Benchmark: single-state vs multi-state design (the result)

Design each backbone **alone** with normal (single-state) MPNN and show those single-state sequences
**fail the other state**. The contrast — multi-state sequences fit *both*, single-state ones fit only
their own — is the headline finding. (Mock: we approximate single-state design by scoring each
sequence against only its designed-for state.)

In [ ]:
# Multi-state: how often does ONE sequence pass BOTH states?
multi_both = float(df["pass_both"].mean())

# Single-state baseline (approximation on mock): design FOR state A only, then test state B.
# A single-state-A sequence is, by construction, optimized for A; we ask how often it ALSO passes B.
# (On real data: run normal MPNN on backbone A alone, predict that sequence toward B, measure pass_b.)
single_a_passes_b = float(df.loc[df["pass_a"], "pass_b"].mean()) if df["pass_a"].any() else float("nan")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["multi-state\n(pass BOTH)", "single-state A\n(also passes B)"],
       [multi_both, single_a_passes_b], color=["tab:green", "tab:gray"])
ax.set_ylabel("fraction"); ax.set_title("Multi-state vs single-state: who satisfies both states?")
for i, v in enumerate([multi_both, single_a_passes_b]):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout(); plt.savefig("results/single_vs_multi.png", dpi=150); plt.show()
print(f"multi-state pass-both rate     : {multi_both:.2f}")
print(f"single-state-A also-passes-B   : {single_a_passes_b:.2f}")
print("Expected (and the point): single-state sequences usually FAIL the other state.")
print("[SYNTHETIC EXAMPLE_DATA — on real data, run normal MPNN per backbone for the true baseline]")

## 4 · Honest hit-rate ladder + the energy-gap caveat

Report the ladder explicitly and sanity-check the switchable picks: is the gap real, or an artifact
of AF2 only seeing one state? A switchable flag survives only if **both** states reproduce when you
bias prediction toward each.

In [ ]:
ladder = dict(
    N_total=len(df),
    N_pass_A=int(df["pass_a"].sum()),
    N_pass_B=int(df["pass_b"].sum()),
    N_pass_BOTH=int(df["pass_both"].sum()),
    N_switchable=int(df["switchable"].fillna(False).sum()),
    N_both_and_switchable=int((df["pass_both"] & df["switchable"].fillna(False)).sum()),
)
for k, v in ladder.items():
    print(f"  {k:24s}: {v}")
print("\nENERGY-GAP CAVEAT (state it in the report):")
print(" - the gap is a relative-units proxy from prediction fit, NOT a kcal/mol free energy;")
print(" - AF2 may only return one state, so a 'switchable' flag is a HYPOTHESIS to test in the wet lab;")
print(" - validate switchable picks by biasing prediction toward EACH state (templates/initial guess).")
print("\n[mock numbers are SYNTHETIC EXAMPLE_DATA]")

## 5 · (Extension) transition-plausibility MD `[extension]`
On a top pick, run a short OpenMM MD (10–50 ns) on each predicted state to check it stays put, and
(harder) probe whether A↔B is plausible. **Not** a free-energy calculation — a sanity probe.

In [ ]:
# Scaffold (fill in with the real OpenMM backend on a GPU runtime):
# 1) prep each predicted state PDB (pdbfixer: add H, solvate, neutralize) at the trigger condition;
# 2) minimize + equilibrate; run 10-50 ns; measure backbone RMSD drift vs the predicted state;
# 3) (stretch) attempt a biased path A->B and report whether the transition looks plausible.
print("Transition-MD scaffold — implement with OpenMM on a GPU runtime (see MANUAL.md §2). "
      "Keep runs short; this probes plausibility, not free energy.")

## D3 (part 2) checklist
- [ ] AF2 predicts **both** states from one sequence; per-state scRMSD plotted; state-biased predictions discussed.
- [ ] Energy-gap distribution with the switchable band; the **gap caveat** stated (not a ΔΔG; AF2 may miss a state).
- [ ] **Single- vs multi-state benchmark**: single-state sequences shown to fail the other state.
- [ ] Honest hit-rate ladder: N(A) / N(B) / N(BOTH) / N(switchable) / N(both & switchable).
- [ ] (ext) transition MD on a top pick.

**Next:** `05_validation_plan.ipynb` — the state-change read-out plan with controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — a state-change read-out that proves a switch

**Standard slot:** *validation plan.* **For Project 22 this means:** turn your top multi-state design
into a **wet-lab read-out** that actually proves the switch (FRET / protease accessibility / SAXS),
with the **paired controls** that make the result interpretable, plus a costed protocol (D4). Stretch:
a LOV light-switch integration (D5-adjacent).

A read-out plan without controls proves nothing. The single-state **"locked" negative** is the
control that distinguishes a real switch from a protein that simply changes shape on its own.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Pick the read-out for your trigger + topology

| Read-out | What it measures | Best when | Key control |
|----------|------------------|-----------|-------------|
| **FRET** | donor/acceptor distance change A↔B | the trigger moves two labeled sites apart/together | a locked single-state mutant (no FRET change) |
| **Protease accessibility** | a site exposed in one state, buried in the other | the switch buries/exposes a cleavable loop | a state-locked variant (constant cleavage) |
| **SAXS** | solution shape / Rg change on trigger | global shape differs between states (hinge, domain-swap) | an unrelated rigid protein (no Rg shift) |

Choose the one whose physical change your two states actually produce, then specify the trigger
titration (pH series / ligand dose / dark-vs-lit).

In [ ]:
READOUT = "FRET"     # FRET | protease | SAXS  -- match it to YOUR two states' physical change
TRIGGER = "pH"       # keep in sync with data/inputs/two_state_def.txt

readout_notes = {
    "FRET":     "label two sites that move apart/together between A and B; report %FRET change vs trigger",
    "protease": "limited proteolysis at a site exposed in one state / buried in the other; report cleavage vs trigger",
    "SAXS":     "SEC-SAXS at each trigger setpoint; report Rg / P(r) change between states",
}
print("read-out:", READOUT, "->", readout_notes[READOUT])
print("trigger  :", TRIGGER)

## 2 · The mandatory controls

Every state-change claim needs three controls. Generate them alongside the design:
- **Positive:** a design / natural protein **known to switch** under this trigger (validates the assay).
- **Single-state "locked" negative:** a sequence engineered to stay in **one** state — your
  always-OFF / always-ON control. If it shows the same read-out change as your switch, your switch
  signal is an artifact.
- **Unrelated:** an unrelated protein of similar size (rules out trigger-induced assay artifacts).

In [ ]:
controls = {
    "positive":   "a known switch (e.g. a LOCKR variant or a natural two-state hinge) under the same trigger",
    "negative_locked": "a single-state 'locked' design (no switch) — the always-OFF / always-ON control",
    "unrelated":  "an unrelated, similarly sized protein with no expected trigger response",
}
for k, v in controls.items():
    print(f"  {k:18s}: {v}")
print("\nThe locked negative is the decisive control: it isolates 'switching' from 'any shape change'.")

## 3 · Generate the costed validation-plan card

Write a concrete, costed protocol the wet lab can execute: expression, purification, the trigger
titration, the read-out, the controls, a timeline, and a reagent list. Fill the `<...>` and `?` from
your top design and local prices.

In [ ]:
design_id = "<your_top_switch_design_id>"   # from results/switch_candidates.csv

plan = f"""# Validation Plan — Project 22 switch {design_id}
# read-out: {READOUT} | trigger: {TRIGGER}

## Construct & expression
- Gene: codon-optimized for E. coli; His6 tag (+ TEV); {READOUT}-specific tags (e.g. CyPet/YPet for FRET).
- Strain: BL21(DE3); 16-18 C overnight induction; IMAC -> SEC purification; confirm monodisperse by SEC.

## Switch assay ({READOUT})
- {readout_notes[READOUT]}.
- Trigger titration: a {TRIGGER} series spanning OFF and ON setpoints, >= 3 replicates per point.
- Primary metric: read-out change between the two trigger extremes (define a pass threshold a priori).

## Controls (run in the SAME plate/session)
- Positive: {controls['positive']}.
- Negative (locked): {controls['negative_locked']}  <-- the decisive control.
- Unrelated: {controls['unrelated']}.

## Timeline (indicative)
- Wk 1-2 cloning + expression test; Wk 3 purification + QC; Wk 4-5 assay + titration; Wk 6 analysis.

## Reagents / cost (fill from local prices)
- gene synthesis x(design + controls) ~ $?; expression/purification consumables ~ $?;
- {READOUT} reagents (labels/protease/beamtime) ~ $?; total ~ $?.

## Go / no-go
- GO if the switch shows a trigger-dependent read-out change that the LOCKED negative does NOT.
"""
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> and ? from your top design + local prices.")
print(plan)

## 4 · (Stretch) LOV light-switch integration `[stretch]`
Make the trigger **light** by fusing/embedding a **LOV photoswitch** (a verified LOV accession from
`data/README.md`). The validation gains **dark vs lit** states: collect the read-out under dark and
blue-light illumination, with the same three controls plus a dark-state baseline.

In [ ]:
# Scaffold for the LOV stretch (design + validate):
# 1) choose a LOV insertion/fusion point so the photo-induced Jalpha change drives YOUR A<->B switch;
# 2) re-run the multi-state design with the LOV module present (it constrains the topology);
# 3) validate with dark vs blue-light (~450 nm) states; controls: a LOV-dead (C->A) photo-insensitive
#    variant (the locked negative) + dark-state baseline.
print("LOV light-switch scaffold — adds a light trigger and a dark/lit validation axis. "
      "Use a verified LOV accession (data/README.md) and a photo-dead variant as the locked control.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md`: read-out (FRET/protease/SAXS) matched to your states + trigger titration.
- [ ] All three controls specified — including the single-state **locked negative** (the decisive one).
- [ ] Expression (BL21(DE3)) + purification + timeline + costed reagent list.
- [ ] The selected multi-state design + its two-state validation attached (D★).
- [ ] (stretch) LOV light-switch integration with dark/lit states + photo-dead control.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; energy-gap + multi-state-designability caveats stated.

You're done — a rigorously characterized **switch hypothesis** with a real plan to test it.